# Stage 3: Hybrid Search -- Combining Semantic + Keyword Precision

In Stage 2, we built a working RAG pipeline with vector search. It grounded
the LLM in real papers -- no more hallucinations. But we hit a wall:
**semantic drift**. When we searched for "p53 mutations in glioblastoma,"
we got papers about cancer genetics in general, not our specific topic.

The fix? Add **keyword search** alongside vector search. In this notebook
we'll:
1. Build a BM25 keyword index over our papers
2. Store both dense vectors and sparse BM25 vectors in Qdrant
3. Fuse the two signals with Reciprocal Rank Fusion (RRF)
4. Show how hybrid search fixes semantic drift
5. Discover what's *still* missing

## 1. Setup + Load Data

In [2]:
import os
import json
import time

from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import models

load_dotenv("../.env")
client = OpenAI()
qdrant = QdrantClient(url=os.getenv("QDRANT_URL", "http://localhost:6333"))

COLLECTION = "pubmed_hybrid"
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM = 1536

with open("../data/pubmed_dataset.json") as f:
    dataset = json.load(f)

# Filter papers with abstracts, take first 5000
papers = [p for p in dataset["papers"] if p.get("abstract")][:5_000]
print(f"Loaded {len(papers)} papers with abstracts")

Loaded 5000 papers with abstracts


## 2. What Is BM25?

BM25 (Best Matching 25) is a classical keyword matching algorithm that's
been the backbone of search engines for decades. Unlike vector search, it
doesn't care about *meaning* -- if the document contains your exact words,
BM25 finds it. It scores documents based on:

- **Term Frequency (TF):** How often does the keyword appear in the document?
- **Inverse Document Frequency (IDF):** How rare is the keyword across all documents?
- **Document length normalization:** Longer documents don't get unfair advantages.

By combining BM25 with vector search, we get both **semantic understanding**
AND **keyword precision**. Vector search handles synonyms and paraphrasing;
BM25 nails the exact terms. Together, they cover each other's blind spots.

## 3. Build BM25 Index

In [17]:
from fastembed import SparseTextEmbedding

# Fit BM25 on all abstracts
corpus = [p["abstract"] for p in papers]
bm25 = SparseTextEmbedding("Qdrant/Bm25")

In [18]:
result = next(iter(bm25.embed("The quick brown fox")))
print(f"Indices: {result.indices[:10]}")
print(f"Values: {result.values[:10]}")

Indices: [ 771291085  741580288 1621867415]
Values: [1.67868852 1.67868852 1.67868852]


## 4. Create Qdrant Collection with Dense + Sparse Vectors

This time we configure the collection with **two** vector types: a dense
vector (from the OpenAI embedding model) and a sparse BM25 vector. Qdrant
stores and searches both, and we can fuse results at query time.

In [19]:
if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config={
        "dense": models.VectorParams(size=EMBED_DIM, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        # why use the IDF modifier? See note in 03_hybrid_search.ipynb for details
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF),
    },
)
print(f"Collection '{COLLECTION}' created (dense + sparse)")

Collection 'pubmed_hybrid' created (dense + sparse)


### Note on BM25 with Qdrant
It may not be immediately clear why 1) "key-words" means storing sparse vectors, and 2) why we need to use a specific "IDF" modifier. The quick answer is that BM25 with Qdrant is actually "BM25-ish" -- its a variant of BM25 adapted to work with sparse vectors offline without having to know the full corpus in advance. The IDF modifier is a way to inject some of the BM25 magic into the sparse vectors ont he server (where we **do** have access to the full corpus) so that we can get better results when we do sparse vector search at query time.

For more infomoration, here is the BM25 Formula used in Qdrant: https://www.elastic.co/blog/practical-bm25-part-2-the-bm25-algorithm-and-its-variables

## 5. Ingest Papers with Both Vectors

For each paper we compute two things: a dense embedding via OpenAI and a
sparse BM25 vector from our tokenizer. Both get stored as named vectors
on the same Qdrant point.

In [23]:
BATCH_SIZE = 100
MAX_TOKENS = 8192
start = time.time()

for i in range(0, len(papers), BATCH_SIZE):
    batch = papers[i : i + BATCH_SIZE]
    texts = [p["abstract"][:MAX_TOKENS] for p in batch]

    # Batch embed with OpenAI
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    dense_vectors = [e.embedding for e in response.data]

    # Build points with both dense and sparse vectors
    points = []
    for paper, dense_vector in zip(batch, dense_vectors):
        result = next(iter(bm25.embed(paper["abstract"])))
        indices = result.indices
        values = result.values
        points.append(
            models.PointStruct(
                id=int(paper["pmid"]),
                vector={
                    "dense": dense_vector,
                    "bm25": models.SparseVector(indices=indices, values=values),
                },
                payload={
                    "pmid": paper["pmid"],
                    "title": paper["title"],
                    "abstract": paper["abstract"],
                    "authors": [a["name"] for a in paper.get("authors", [])],
                    "journal": paper.get("journal", ""),
                    "mesh_terms": [m["term"] for m in paper.get("mesh_terms", [])],
                },
            )
        )

    qdrant.upsert(collection_name=COLLECTION, points=points)
    print(f"  Ingested {min(i + BATCH_SIZE, len(papers))}/{len(papers)} papers")

elapsed = time.time() - start
print(f"\nDone! {len(papers)} papers ingested in {elapsed:.1f}s")

  Ingested 100/5000 papers
  Ingested 200/5000 papers
  Ingested 300/5000 papers
  Ingested 400/5000 papers
  Ingested 500/5000 papers
  Ingested 600/5000 papers
  Ingested 700/5000 papers
  Ingested 800/5000 papers
  Ingested 900/5000 papers
  Ingested 1000/5000 papers
  Ingested 1100/5000 papers
  Ingested 1200/5000 papers
  Ingested 1300/5000 papers
  Ingested 1400/5000 papers
  Ingested 1500/5000 papers
  Ingested 1600/5000 papers
  Ingested 1700/5000 papers
  Ingested 1800/5000 papers
  Ingested 1900/5000 papers
  Ingested 2000/5000 papers
  Ingested 2100/5000 papers
  Ingested 2200/5000 papers
  Ingested 2300/5000 papers
  Ingested 2400/5000 papers
  Ingested 2500/5000 papers
  Ingested 2600/5000 papers
  Ingested 2700/5000 papers
  Ingested 2800/5000 papers
  Ingested 2900/5000 papers
  Ingested 3000/5000 papers
  Ingested 3100/5000 papers
  Ingested 3200/5000 papers
  Ingested 3300/5000 papers
  Ingested 3400/5000 papers
  Ingested 3500/5000 papers
  Ingested 3600/5000 papers
 

## 6. Hybrid Search: Dense + BM25 with Fusion

Now we can search with both signals. Qdrant's **prefetch + fusion** lets us
retrieve from each index independently, then combine them using Reciprocal
Rank Fusion (RRF). RRF is elegantly simple: it ranks each result by
`1 / (k + rank)` in each retriever, then sums the scores. Papers that rank
well in *both* dense and BM25 float to the top.

In [24]:
def hybrid_search(question, top_k=5):
    """
    Hybrid search: dense vectors + BM25, fused by RRF.
    """
    # Dense query vector
    q_embedding = client.embeddings.create(
        model=EMBED_MODEL, input=question
    ).data[0].embedding

    # BM25 sparse query vector
    bm25_result = next(iter(bm25.embed(question)))
    bm25_indices = bm25_result.indices
    bm25_values = bm25_result.values

    results = qdrant.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(query=q_embedding, using="dense", limit=top_k * 3),
            models.Prefetch(
                query=models.SparseVector(indices=bm25_indices, values=bm25_values),
                using="bm25",
                limit=top_k * 3,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k,
        with_payload=True,
    )
    return results.points

## 7. Compare Dense-Only vs Hybrid

Let's revisit the query from Stage 2 that tripped up pure vector search.
We asked about **p53 mutations in glioblastoma** and got semantically
related but imprecise results. Does adding BM25 fix it?

In [26]:
specific_question = "What papers discuss p53 tumor suppressor gene mutations and the q13.r region specifically in glioblastoma multiforme?"

# We need the dense embedding for the dense-only comparison
q_embedding = client.embeddings.create(
    model=EMBED_MODEL, input=specific_question
).data[0].embedding

# Dense only (same approach as Stage 2)
dense_results = qdrant.query_points(
    collection_name=COLLECTION,
    query=q_embedding,
    using="dense",
    limit=5,
    with_payload=True,
).points

print("=== Dense Only (Stage 2 approach) ===\n")
for i, r in enumerate(dense_results, 1):
    print(f"{i}. {r.payload['title'][:80]}")
    print(f"   Abstract: {r.payload['abstract'][:200]}...")
    print(f"   MeSH: {', '.join(r.payload.get('mesh_terms', [])[:4])}\n")

print("\n=== Hybrid: Dense + BM25 (Stage 3) ===\n")
hybrid_results = hybrid_search(specific_question)
for i, r in enumerate(hybrid_results, 1):
    print(f"{i}. {r.payload['title'][:80]}")
    print(f"   Abstract: {r.payload['abstract'][:200]}...")
    print(f"   MeSH: {', '.join(r.payload.get('mesh_terms', [])[:4])}\n")

=== Dense Only (Stage 2 approach) ===

1. In vivo Engineering of Chromosome 19 q-arm by Employing the CRISPR/AsCpf1 and dd
   Abstract: Deletions of the q13.3 region of chromosome 19 have been found commonly in all three main kinds of diffuse human malignant gliomas, powerfully demonstrating the existence of tumor suppressor genes in ...
   MeSH: Bacterial Proteins, Brain Neoplasms, Chromosome Deletion, Chromosomes, Human, Pair 19

2. Genome-wide CRISPR screens identify novel regulators of wild-type and mutant p53
   Abstract: Tumor suppressor p53 (TP53) is frequently mutated in cancer, often resulting not only in loss of its tumor-suppressive function but also acquisition of dominant-negative and even oncogenic gain-of-fun...
   MeSH: Tumor Suppressor Protein p53, Animals, Humans, Mice

3. Deep CRISPR mutagenesis characterizes the functional diversity of TP53 mutations
   Abstract: The mutational landscape of TP53, a tumor suppressor mutated in about half of all cancers, includes over

Notice the difference. The hybrid results should favor papers that
actually contain the words "p53," "glioblastoma," and "mutations" -- not
just papers that live in the same semantic neighborhood as cancer genetics.
BM25 gives keyword precision; dense vectors give semantic coverage. RRF
combines the best of both.

## 8. RAG with Hybrid Retrieval

In [27]:
def rag_answer(question, results):
    """Generate an answer using retrieved papers as context."""
    context = "\n\n".join(
        [
            f"Paper {i+1} (PMID: {r.payload['pmid']}):\n"
            f"Title: {r.payload['title']}\n"
            f"Abstract: {r.payload['abstract'][:500]}"
            for i, r in enumerate(results)
        ]
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a biomedical research assistant. Answer questions "
                    "using ONLY the provided paper context. Cite papers by PMID."
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}",
            },
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content

In [28]:
answer = rag_answer(specific_question, hybrid_results)
print(answer)

The paper that discusses the q13.3 region specifically in glioblastoma multiforme is Paper 1 (PMID: 33990905), which highlights deletions in this region and the presence of tumor suppressor genes. Additionally, Paper 2 (PMID: 38580884) discusses p53 mutations but does not specifically mention the q13.3 region or glioblastoma multiforme. Therefore, only Paper 1 directly addresses the q13.3 region in the context of glioblastoma.


## 9. The Gap

Hybrid search gives us the best of both worlds: semantic understanding +
keyword precision. But there's still something missing.

Think about how a **real literature review** works. You don't just find
similar papers -- you **chase citations**. You find the foundational papers
that everyone in the field builds on. A paper might be semantically distant
from your query (it's a general methods paper from 2012) but it's cited by
*every single paper* you retrieved.

**Qdrant tells you what's *similar*. But it can't tell you what's
*connected*.**

A vector database treats each paper as an isolated point in space. It has
no concept of "Paper A cites Paper B" or "these five papers all build on
the same foundational work." Citation networks, co-authorship graphs,
shared MeSH term hierarchies -- all of this relational structure is
invisible to a vector index.

For that, we need a **graph database**.

## 10. Architecture

```
Stage 3 architecture:
    Question --> [Qdrant: Dense + BM25 + RRF] --> Papers --> [LLM] --> Answer

Improvement: Keyword precision eliminates semantic drift

Next gap: Can't find foundational/connected papers (citation lineage)
```

In the next notebook, we add a graph layer to capture the connections
between papers -- and unlock a whole new class of queries.